# TA-03: Klasifikasi Kegagalan Mesin Industri dengan Logistic Regression
Dataset: AI4I 2020 Predictive Maintenance (UCI #601) – 10.000 baris, target `Machine failure`.

## 1. Import

In [ ]:
import os, io, zipfile, urllib.request
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns, joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_curve, roc_auc_score,
                             precision_recall_curve)
from imblearn.over_sampling import SMOTE
from features import build_features, FEATURES

## 2. Load data

In [ ]:
CSV = 'ai4i2020.csv'
if not os.path.exists(CSV):
    url = 'https://archive.ics.uci.edu/static/public/601/ai4i+2020+predictive+maintenance+dataset.zip'
    try:
        z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read()))
        z.extract('ai4i2020.csv', '.')
    except Exception as e:
        raise SystemExit(f'Unduh manual dari halaman UCI, taruh ai4i2020.csv di folder ini. ({e})')
df = pd.read_csv(CSV)
print(df.shape); df.head()

## 3. EDA

In [ ]:
df.info(); print(df.isna().sum().sum(), 'missing values'); df.describe()

## 3a. Distribusi target (class imbalance)

In [ ]:
print(df['Machine failure'].value_counts(normalize=True))
sns.countplot(x='Machine failure', data=df); plt.title('Distribusi Machine failure'); plt.show()

## 3b. Distribusi fitur vs target

In [ ]:
num_cols = ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]',
            'Torque [Nm]', 'Tool wear [min]']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, c in zip(axes.ravel(), num_cols):
    sns.boxplot(x='Machine failure', y=c, data=df, ax=ax)
sns.countplot(x='Type', hue='Machine failure', data=df, ax=axes[1, 2])
plt.tight_layout(); plt.show()

## 3c. WASPADA DATA LEAKAGE: kolom mode kegagalan menentukan target

In [ ]:
modes = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
print('Machine failure == 1 bila salah satu mode = 1?',
      ((df[modes].sum(axis=1) > 0).astype(int) == df['Machine failure']).mean())
print('Ini bocoran target -> kolom', modes, 'WAJIB dibuang dari fitur.')

## 4. Preprocessing

In [ ]:
d = df.rename(columns={'Type': 'type', 'Air temperature [K]': 'air_temp',
                       'Process temperature [K]': 'process_temp',
                       'Rotational speed [rpm]': 'rpm', 'Torque [Nm]': 'torque',
                       'Tool wear [min]': 'tool_wear', 'Machine failure': 'failure'})
d = d.drop(columns=['UDI', 'Product ID'] + modes)       # ID & kolom bocoran
X, y = build_features(d), d['failure']
print(X.head())
sns.heatmap(pd.concat([X, y], axis=1).corr(), annot=True, fmt='.2f', cmap='coolwarm'); plt.show()

## 5. Split, scaling, SMOTE (hanya pada data latih!)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler().fit(X_train)
Xtr, Xte = scaler.transform(X_train), scaler.transform(X_test)
Xtr_sm, ytr_sm = SMOTE(random_state=42).fit_resample(Xtr, y_train)
print('Sebelum SMOTE:', y_train.value_counts().to_dict(), '| Sesudah:', ytr_sm.value_counts().to_dict())

## 6. Training: baseline (tanpa SMOTE) vs SMOTE

In [ ]:
base = LogisticRegression(max_iter=1000, random_state=42).fit(Xtr, y_train)
model = LogisticRegression(max_iter=1000, random_state=42).fit(Xtr_sm, ytr_sm)

## 7. Evaluasi

In [ ]:
def evaluate(m, name, thr=0.5):
    proba = m.predict_proba(Xte)[:, 1]
    pred = (proba >= thr).astype(int)
    print(f'===== {name} (threshold={thr}) =====')
    print(classification_report(y_test, pred, target_names=['Normal', 'Gagal']))
    ConfusionMatrixDisplay(confusion_matrix(y_test, pred),
                           display_labels=['Normal', 'Gagal']).plot(cmap='Blues')
    plt.title(f'Confusion Matrix - {name}'); plt.show()
    print('AUC:', round(roc_auc_score(y_test, proba), 4)); return proba

p_base = evaluate(base, 'Baseline tanpa SMOTE')
p_sm = evaluate(model, 'Dengan SMOTE')

## 7a. Kurva ROC

In [ ]:
for p, n in [(p_base, 'Baseline'), (p_sm, 'SMOTE')]:
    fpr, tpr, _ = roc_curve(y_test, p); plt.plot(fpr, tpr, label=f'{n} (AUC={roc_auc_score(y_test, p):.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Tebakan acak'); plt.xlabel('FPR'); plt.ylabel('TPR')
plt.legend(); plt.grid(True); plt.title('ROC Curve'); plt.show()

## 7b. Trade-off threshold (precision vs recall)

In [ ]:
prec, rec, thr = precision_recall_curve(y_test, p_sm)
plt.plot(thr, prec[:-1], label='Precision'); plt.plot(thr, rec[:-1], label='Recall')
plt.xlabel('Threshold'); plt.legend(); plt.grid(True); plt.title('Precision-Recall vs Threshold'); plt.show()

## 8. Interpretasi koefisien (fitur sudah distandarisasi)

In [ ]:
coef = pd.DataFrame({'fitur': FEATURES, 'koefisien': model.coef_[0],
                     'odds_ratio_per_1SD': np.exp(model.coef_[0])}).sort_values('koefisien', key=abs, ascending=False)
print(coef)

## 9. Simpan model & scaler

In [ ]:
joblib.dump(model, 'model.pkl'); joblib.dump(scaler, 'scaler.pkl')
print('Tersimpan: model.pkl, scaler.pkl')